# Predictive Power — per-feature univariate AUC, NEW vs OLD

`PredictivePower` removes the masking — for each
feature it trains a tiny one-column LightGBM against the target, so
`auc_new - auc_old` directly answers: **did the fix make this feature a
better feature?**

Two runs, by design:

1. **Percent features, all three bureaus combined (train)** — the
   denominator fix moves percent features the same way on every bureau, so
   pooling 1.2M applicants gives maximum power on the original fix.
2. **Number features, experian only (train)** — number features only change
   under the experian placeholder change (eq/TU numerators are untouched, so
   pooling would just dilute the signal with identical data). This isolates
   the placeholder change's contribution.

NEW for experian = `new_normalized_and_processed` (placeholder change
included); eq/tu = `processed_new`. OLD = `processed_old`. Same applicants
on both sides, so any AUC delta is attributable to the data change alone.
Treat deltas within ~±0.002 as noise. Model-engine kernel.

In [1]:
import glob
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from zaml.analyze.data_analysis.predictive_power import PredictivePower

DATA    = '/home/jag/payment-processor-research/payment_processing_research_data'
BUREAUS = ['equifax', 'experian', 'transunion']
TARGET  = 'final_DQ60_m24'
N_JOBS  = 16

def processed_dir(variant, bureau):
    if variant == 'new' and bureau == 'experian':
        return f'{DATA}/new_normalized_and_processed/experian_train/processed'
    return f'{DATA}/samples/{bureau}_train/processed_{variant}'

# feature lists from the schema
schema_file = sorted(glob.glob(f'{DATA}/samples/transunion_train/processed_new/part-*.parquet'))[0]
cols = pq.read_schema(schema_file).names
percent_features = [c for c in cols if 'percent_of_DQ' in c and 'in_last' in c.lower()]
number_features  = [c for c in cols if 'number_of_DQ' in c and 'in_last' in c.lower()]
print(f'{len(percent_features)} percent features, {len(number_features)} number features')

/home/jag/.conda/envs/model_engine_2_py310/lib/python3.10/site-packages/zaml/common/utils/io.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


1143 percent features, 1208 number features


In [2]:
# targets for the train samples, indexed by ZEST_KEY
targets = pd.concat([
    pd.read_parquet(f'{DATA}/samples/{b}_train/target.parquet', columns=['ZEST_KEY', TARGET])
    for b in BUREAUS], ignore_index=True).dropna(subset=[TARGET])
targets = targets.drop_duplicates('ZEST_KEY').set_index('ZEST_KEY')[TARGET]
print(f'targets: {len(targets):,} | bad rate: {targets.mean():.4f}')

targets: 1,200,000 | bad rate: 0.0761


In [3]:
def pp_compare(new, old, y, features, label, n_jobs=N_JOBS):
    """Univariate AUC per feature, NEW vs OLD, same applicants + target."""
    idx = new.index.intersection(old.index).intersection(y.index)
    yy = y.loc[idx]
    print(f'{label}: {len(idx):,} applicants, bad rate {yy.mean():.4f}')

    pp_new = PredictivePower(n_jobs=n_jobs, seed=0).calculate(new.loc[idx, features], yy)
    pp_old = PredictivePower(n_jobs=n_jobs, seed=0).calculate(old.loc[idx, features], yy)

    cmp = (pp_new[['auc']].rename(columns={'auc': 'auc_new'})
           .join(pp_old[['auc']].rename(columns={'auc': 'auc_old'})))
    cmp['new_minus_old'] = cmp['auc_new'] - cmp['auc_old']
    cmp = cmp.sort_values('new_minus_old', ascending=False)

    print(f'mean delta: {cmp["new_minus_old"].mean():+.5f} | '
          f'features improved: {(cmp["new_minus_old"] > 0).sum()}/{len(cmp)} | '
          f'|delta| > 0.002: {(cmp["new_minus_old"].abs() > 0.002).sum()}')
    print('\ntop 15 improvements:')
    print(cmp.head(15).round(4).to_string())
    print('\ntop 10 degradations:')
    print(cmp.tail(10).round(4).to_string())
    return cmp

In [5]:
# 1) PERCENT features -- all bureaus combined (train), 1.2M applicants
new_all = pd.concat([pd.read_parquet(processed_dir('new', b), columns=percent_features)
                     for b in BUREAUS])
old_all = pd.concat([pd.read_parquet(processed_dir('old', b), columns=percent_features)
                     for b in BUREAUS])
pp_percent = pp_compare(new_all, old_all, targets, percent_features,
                        'PERCENT features, all bureaus')
del new_all, old_all

PERCENT features, all bureaus: 1,184,475 applicants, bad rate 0.0747
mean delta: +0.00014 | features improved: 712/1143 | |delta| > 0.002: 30

top 15 improvements:
                                                                                                        auc_new  auc_old  new_minus_old
trade_max_percent_of_DQ30_in_last_24_months__with_historic_utilization_over_50_percent_open_cc           0.6441   0.6401         0.0040
trade_max_percent_of_DQ30_in_last_24_months__with_historic_utilization_over_25_percent_open_cc           0.6330   0.6291         0.0039
trade_max_percent_of_DQ30_in_last_24_months__all_open_cc                                                 0.6305   0.6269         0.0036
trade_max_percent_of_DQ30_in_last_24_months__individual_open_cc                                          0.6344   0.6308         0.0036
trade_max_percent_of_DQ30_in_last_24_months__active_open_cc                                              0.6306   0.6271         0.0036
trade_max_percent_of

In [17]:
pp_percent['new_minus_old'].describe()

count    1143.000000
mean        0.000142
std         0.000571
min        -0.001488
25%        -0.000012
50%         0.000006
75%         0.000066
max         0.003986
Name: new_minus_old, dtype: float64

In [13]:
pp_percent.sort_values(by = 'new_minus_old', ascending = False)

,auc_new,auc_old,new_minus_old
trade_max_percent_of_DQ30_in_last_24_months__with_historic_utilization_over_50_percent_open_cc,0.644101,0.640115,0.003986
trade_max_percent_of_DQ30_in_last_24_months__with_historic_utilization_over_25_percent_open_cc,0.632998,0.629065,0.003933
trade_max_percent_of_DQ30_in_last_24_months__all_open_cc,0.630515,0.626907,0.003609
trade_max_percent_of_DQ30_in_last_24_months__individual_open_cc,0.634370,0.630775,0.003595
trade_max_percent_of_DQ30_in_last_24_months__active_open_cc,0.630624,0.627073,0.003552
...,...,...,...
trade_max_percent_of_DQ30_or_greater_in_last_6_months__individual_open_installment,0.555018,0.556300,-0.001282
trade_mean_percent_of_DQ30_or_greater_in_last_6_months__active_open_installment,0.551934,0.553398,-0.001465
trade_max_percent_of_DQ30_or_greater_in_last_6_months__active_open_installment,0.551831,0.553305,-0.001473
trade_mean_percent_of_DQ30_or_greater_in_last_6_months__all_open_installment,0.551892,0.553366,-0.001474


In [7]:
# 2) NUMBER features -- experian only (train)
exp_new = pd.read_parquet(processed_dir('new', 'experian'), columns=number_features)
exp_old = pd.read_parquet(processed_dir('old', 'experian'), columns=number_features)
pp_number = pp_compare(exp_new, exp_old, targets, number_features,
                       'NUMBER features, experian only')

NUMBER features, experian only: 394,241 applicants, bad rate 0.0805
mean delta: -0.00006 | features improved: 504/1208 | |delta| > 0.002: 0

top 15 improvements:
                                                                                       auc_new  auc_old  new_minus_old
trade_mean_number_of_DQ30_in_last_24_months__active_open_accounts                       0.6125   0.6116         0.0009
trade_mean_number_of_DQ60_in_last_12_months__active_open_installment                    0.5194   0.5187         0.0007
trade_mean_number_of_DQ60_in_last_24_months__derog_open_charge_card                     0.6552   0.6548         0.0004
trade_mean_number_of_DQ30_in_last_24_months__with_recent_payment_open_accounts          0.6212   0.6208         0.0004
trade_mean_number_of_DQ30_in_last_12_months__individual_open_revolving                  0.6468   0.6464         0.0004
trade_mean_number_of_DQ60_in_last_24_months__joint_open_accounts                        0.6176   0.6173         0.0004
trade

In [14]:
pp_number.sort_values(by = 'new_minus_old', ascending = False)

,auc_new,auc_old,new_minus_old
trade_mean_number_of_DQ30_in_last_24_months__active_open_accounts,0.612496,0.611625,0.000871
trade_mean_number_of_DQ60_in_last_12_months__active_open_installment,0.519380,0.518696,0.000684
trade_mean_number_of_DQ60_in_last_24_months__derog_open_charge_card,0.655240,0.654836,0.000404
trade_mean_number_of_DQ30_in_last_24_months__with_recent_payment_open_accounts,0.621183,0.620791,0.000392
trade_mean_number_of_DQ30_in_last_12_months__individual_open_revolving,0.646819,0.646429,0.000390
...,...,...,...
trade_sum_number_of_DQ30_in_last_24_months__all_open_charge_card,0.621149,0.622597,-0.001447
trade_max_number_of_DQ30_in_last_24_months__individual_open_charge_card,0.619792,0.621283,-0.001492
trade_max_number_of_DQ30_in_last_24_months__all_open_charge_card,0.619871,0.621393,-0.001522
trade_mean_number_of_DQ60_in_last_24_months__individual_open_charge_card,0.618592,0.620282,-0.001690


In [15]:
pp_number['new_minus_old'].mean()

-6.037964699728938e-05

In [18]:
pp_number['new_minus_old'].describe()

count    1208.000000
mean       -0.000060
std         0.000213
min        -0.001745
25%        -0.000026
50%         0.000000
75%         0.000004
max         0.000871
Name: new_minus_old, dtype: float64

In [16]:
# sharp version: experian number features on ONLY the applicants whose
# number features actually changed (the placeholder change population)
changed = np.zeros(len(exp_new.index.intersection(exp_old.index)), dtype=bool)
idx = exp_new.index.intersection(exp_old.index)
n, o = exp_new.loc[idx], exp_old.loc[idx]
for c in number_features:
    changed |= ~np.isclose(n[c].astype('float64'), o[c].astype('float64'), equal_nan=True)
print(f'experian applicants with >=1 changed number feature: {changed.sum():,} ({changed.mean():.2%})')

if changed.sum() >= 2000:
    pp_number_changed = pp_compare(n.loc[changed], o.loc[changed], targets,
                                   number_features, 'NUMBER features, experian CHANGED rows')
else:
    print('too few changed applicants for a stable per-feature AUC -- skipping')

experian applicants with >=1 changed number feature: 5,163 (1.31%)
NUMBER features, experian CHANGED rows: 5,163 applicants, bad rate 0.1683
mean delta: +0.00061 | features improved: 583/1208 | |delta| > 0.002: 412

top 15 improvements:
                                                                                   auc_new  auc_old  new_minus_old
trade_mean_number_of_DQ30_in_last_24_months__individual_open_charge_card            0.5831   0.5462         0.0369
trade_sum_number_of_DQ60_in_last_24_months__all_open_accounts                       0.5548   0.5219         0.0329
trade_max_number_of_DQ60_in_last_24_months__with_recent_payment_open_accounts       0.5778   0.5464         0.0314
trade_mean_number_of_DQ30_in_last_12_months__with_recent_payment_open_installment   0.5669   0.5403         0.0267
trade_sum_number_of_DQ60_in_last_24_months__individual_open_accounts                0.5519   0.5259         0.0259
trade_sum_number_of_DQ30_in_last_24_months__all_open_revolving           

In [21]:
pp_number_changed['new_minus_old'].describe()

count    1208.000000
mean        0.000614
std         0.005730
min        -0.041884
25%        -0.000055
50%         0.000000
75%         0.001508
max         0.036937
Name: new_minus_old, dtype: float64